In [ ]:
import pandas as pd
import numpy as np

import sqlite3

import matplotlib.pyplot as plt

from datetime import datetime
np.random.seed(42)

n = 100000

transactions = pd.DataFrame({

    'transaction_id': range(1, n + 1),

    'customer_id': np.random.randint(
        10001,
        30001,
        n
    ),

    'merchant_id': np.random.randint(
        1001,
        3001,
        n
    ),

    'transaction_date': pd.to_datetime(
        np.random.choice(
            pd.date_range(
                '2025-01-01',
                '2025-12-31',
                freq='h'
            ),
            n
        )
    ),

    'amount': np.round(
        np.random.lognormal(
            mean=7,
            sigma=1,
            size=n
        ),
        2
    ),

    'payment_method': np.random.choice(
        [
            'UPI',
            'Credit Card',
            'Debit Card',
            'Net Banking',
            'Wallet'
        ],
        n,
        p=[
            0.45,
            0.20,
            0.20,
            0.10,
            0.05
        ]
    ),

    'bank': np.random.choice(
        [
            'HDFC',
            'ICICI',
            'SBI',
            'Axis',
            'Kotak',
            'Yes Bank'
        ],
        n
    ),

    'card_type': np.random.choice(
        [
            'Visa',
            'Mastercard',
            'RuPay',
            'Not Applicable'
        ],
        n
    ),

    'device': np.random.choice(
        [
            'Mobile',
            'Desktop',
            'Tablet'
        ],
        n,
        p=[
            0.70,
            0.25,
            0.05
        ]
    ),

    'city': np.random.choice(
        [
            'Delhi',
            'Mumbai',
            'Bangalore',
            'Hyderabad',
            'Chennai',
            'Pune',
            'Kolkata',
            'Ahmedabad'
        ],
        n
    )
})
transactions.head()
transactions['status'] = np.random.choice(
    [
        'SUCCESS',
        'FAILED',
        'DECLINED',
        'TIMEOUT'
    ],
    n,
    p=[
        0.88,
        0.06,
        0.04,
        0.02
    ]
)
failure_reasons = [
    'Insufficient Funds',
    'Bank Server Down',
    'Technical Error',
    'Invalid Card',
    'Fraud Suspected',
    'Transaction Limit Exceeded',
    'Network Timeout'
]
transactions['failure_reason'] = np.where(

    transactions['status'] == 'SUCCESS',

    'No Failure',

    np.random.choice(
        failure_reasons,
        n
    )
)
transactions['processing_time'] = np.round(
    np.random.uniform(
        0.5,
        10,
        n
    ),
    2
)
transactions.shape
transactions.columns
transactions.head()
transactions.isnull().sum()
transactions['transaction_id'].duplicated().sum()
transactions.duplicated().sum()
(transactions['amount'] < 0).sum()
transactions['status'].value_counts()
transactions['amount'].describe()
transactions['payment_method'].value_counts()
transactions['bank'].value_counts()
conn = sqlite3.connect(
    'fintech_payments.db'
)
transactions.to_sql(
    'transactions',
    conn,
    if_exists='replace',
    index=False
)
query = """
SELECT COUNT(*) AS total_transactions
FROM transactions;
"""

pd.read_sql_query(
    query,
    conn
)

,total_transactions
0,100000


In [ ]:
query = """
SELECT
    status,
    COUNT(*) AS transaction_count
FROM transactions
GROUP BY status
ORDER BY transaction_count DESC;
"""

status_analysis = pd.read_sql_query(
    query,
    conn
)

status_analysis
query = """
SELECT

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN status = 'SUCCESS'
            THEN 1
            ELSE 0
        END
    ) AS successful_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status = 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS success_rate

FROM transactions;
"""

success_rate = pd.read_sql_query(
    query,
    conn
)

success_rate
query = """
SELECT

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status IN
                ('FAILED','DECLINED','TIMEOUT')
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions;
"""

pd.read_sql_query(
    query,
    conn
)

,failure_rate
0,12.08


In [ ]:
query = """
SELECT

    payment_method,

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN status != 'SUCCESS'
            THEN 1
            ELSE 0
        END
    ) AS failed_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY payment_method

ORDER BY failure_rate DESC;
"""

payment_failure = pd.read_sql_query(
    query,
    conn
)

payment_failure

,payment_method,total_transactions,failed_transactions,failure_rate
0,Debit Card,20146,2447,12.15
1,UPI,44978,5455,12.13
2,Credit Card,19929,2403,12.06
3,Net Banking,9985,1199,12.01
4,Wallet,4962,574,11.57


In [ ]:
query = """
SELECT

    bank,

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN status != 'SUCCESS'
            THEN 1
            ELSE 0
        END
    ) AS failed_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY bank

ORDER BY failure_rate DESC;
"""

bank_failure = pd.read_sql_query(
    query,
    conn
)

bank_failure

,bank,total_transactions,failed_transactions,failure_rate
0,Kotak,16631,2057,12.37
1,Axis,16700,2045,12.25
2,Yes Bank,16797,2033,12.10
3,ICICI,16634,2010,12.08
4,HDFC,16621,1968,11.84
5,SBI,16617,1965,11.83


In [ ]:
query = """
SELECT

    failure_reason,

    COUNT(*) AS failed_transactions

FROM transactions

WHERE status != 'SUCCESS'

GROUP BY failure_reason

ORDER BY failed_transactions DESC;
"""

failure_reason_analysis = pd.read_sql_query(
    query,
    conn
)

failure_reason_analysis

,failure_reason,failed_transactions
0,Network Timeout,1743
1,Invalid Card,1735
2,Insufficient Funds,1731
3,Technical Error,1726
4,Fraud Suspected,1723
5,Bank Server Down,1714
6,Transaction Limit Exceeded,1706


In [ ]:
query = """
SELECT

    SUM(amount) AS failed_transaction_value

FROM transactions

WHERE status != 'SUCCESS';
"""

failed_value = pd.read_sql_query(
    query,
    conn
)

failed_value

,failed_transaction_value
0,21196537.2


In [ ]:
query = """
SELECT

    strftime('%Y-%m', transaction_date)
    AS month,

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN status != 'SUCCESS'
            THEN 1
            ELSE 0
        END
    ) AS failed_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY month

ORDER BY month;
"""

monthly_failure = pd.read_sql_query(
    query,
    conn
)

monthly_failure

,month,total_transactions,failed_transactions,failure_rate
0,2025-01,8571,1069,12.47
1,2025-02,7577,903,11.92
2,2025-03,8515,1084,12.73
3,2025-04,8080,946,11.71
4,2025-05,8528,1027,12.04
5,2025-06,8256,1021,12.37
6,2025-07,8526,1026,12.03
7,2025-08,8590,1001,11.65
8,2025-09,8269,1021,12.35
9,2025-10,8655,1013,11.70


In [ ]:
query = """
SELECT

    city,

    COUNT(*) AS total_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY city

ORDER BY failure_rate DESC;
"""

city_failure = pd.read_sql_query(
    query,
    conn
)
display(city_failure)

,city,total_transactions,failure_rate
0,Delhi,12557,12.38
1,Chennai,12479,12.32
2,Ahmedabad,12399,12.28
3,Kolkata,12298,12.21
4,Bangalore,12704,12.07
5,Hyderabad,12358,12.02
6,Pune,12577,11.75
7,Mumbai,12628,11.59


In [ ]:
query = """
SELECT

    merchant_id,

    COUNT(*) AS failed_transactions,

    SUM(amount) AS failed_transaction_value

FROM transactions

WHERE status != 'SUCCESS'

GROUP BY merchant_id

ORDER BY failed_transactions DESC

LIMIT 10;
"""

top_failed_merchants = pd.read_sql_query(
    query,
    conn
)

top_failed_merchants

,merchant_id,failed_transactions,failed_transaction_value
0,1905,18,41087.50
1,1992,16,18887.67
2,2513,15,44494.74
3,2155,14,16590.04
4,2124,14,25797.69
5,2023,14,14758.40
6,1844,14,15418.02
7,1768,14,24114.28
8,1654,14,20984.98
9,1168,14,44203.22


In [ ]:
query = """
SELECT *

FROM transactions

WHERE status != 'SUCCESS'

AND amount > 50000

ORDER BY amount DESC;
"""

high_value_failures = pd.read_sql_query(
    query,
    conn
)
display(high_value_failures)

,transaction_id,customer_id,merchant_id,transaction_date,amount,payment_method,bank,card_type,device,city,status,failure_reason,processing_time
0,19680,12780,1907,2025-03-31 07:00:00,64585.17,Debit Card,Yes Bank,RuPay,Desktop,Bangalore,DECLINED,Network Timeout,7.74


In [ ]:
query = """
SELECT

    device,

    COUNT(*) AS total_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY device

ORDER BY failure_rate DESC;
"""

device_failure = pd.read_sql_query(
    query,
    conn
)
display (device_failure)

,device,total_transactions,failure_rate
0,Mobile,70037,12.14
1,Tablet,5062,12.11
2,Desktop,24901,11.90


In [ ]:
query = """
SELECT

    card_type,

    COUNT(*) AS total_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY card_type

ORDER BY failure_rate DESC;
"""

card_failure = pd.read_sql_query(
    query,
    conn
)
display(card_failure)

,card_type,total_transactions,failure_rate
0,Mastercard,25170,12.39
1,Not Applicable,24931,12.12
2,RuPay,24915,11.95
3,Visa,24984,11.84


In [ ]:
query = """
SELECT

    bank,

    payment_method,

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN status != 'SUCCESS'
            THEN 1
            ELSE 0
        END
    ) AS failed_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY
    bank,
    payment_method

HAVING COUNT(*) >= 100

ORDER BY failure_rate DESC;
"""

bank_method_analysis = pd.read_sql_query(
    query,
    conn
)
display(bank_method_analysis)

,bank,payment_method,total_transactions,failed_transactions,failure_rate
0,Kotak,Debit Card,3324,440,13.24
1,HDFC,Wallet,853,112,13.13
2,Axis,Debit Card,3349,426,12.72
3,Kotak,Net Banking,1609,203,12.62
4,Kotak,Wallet,848,107,12.62
5,ICICI,Net Banking,1660,209,12.59
6,ICICI,Debit Card,3377,424,12.56
7,Axis,UPI,7540,940,12.47
8,Yes Bank,UPI,7618,950,12.47
9,Yes Bank,Credit Card,3352,415,12.38


In [ ]:
query = """
WITH bank_metrics AS (

    SELECT

        bank,

        COUNT(*) AS total_transactions,

        ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN status != 'SUCCESS'
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS failure_rate

    FROM transactions

    GROUP BY bank
)

SELECT

    bank,

    total_transactions,

    failure_rate,

    RANK() OVER (
        ORDER BY failure_rate DESC
    ) AS failure_rank

FROM bank_metrics;
"""

bank_ranking = pd.read_sql_query(
    query,
    conn
)
display(bank_ranking)

,bank,total_transactions,failure_rate,failure_rank
0,Kotak,16631,12.37,1
1,Axis,16700,12.25,2
2,Yes Bank,16797,12.10,3
3,ICICI,16634,12.08,4
4,HDFC,16621,11.84,5
5,SBI,16617,11.83,6


In [ ]:
query = """
SELECT

    strftime(
        '%H',
        transaction_date
    ) AS hour,

    COUNT(*) AS total_transactions,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN status != 'SUCCESS'
                THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS failure_rate

FROM transactions

GROUP BY hour

ORDER BY failure_rate DESC;
"""

hourly_failure = pd.read_sql_query(
    query,
    conn
)
display(hourly_failure)

,hour,total_transactions,failure_rate
0,12,4119,13.18
1,10,4157,13.16
2,06,4063,12.48
3,22,4081,12.42
4,18,4059,12.27
5,05,4205,12.25
6,07,4121,12.21
7,09,4144,12.19
8,08,4160,12.16
9,04,4206,12.15


In [ ]:
output_file = 'FinTech_Payment_Analytics.xlsx'

In [ ]:
with pd.ExcelWriter(
    output_file,
    engine='openpyxl'
) as writer:

    status_analysis.to_excel(
        writer,
        sheet_name='Status Analysis',
        index=False
    )

    payment_failure.to_excel(
        writer,
        sheet_name='Payment Method',
        index=False
    )

    bank_failure.to_excel(
        writer,
        sheet_name='Bank Analysis',
        index=False
    )

    failure_reason_analysis.to_excel(
        writer,
        sheet_name='Failure Reasons',
        index=False
    )

    monthly_failure.to_excel(
        writer,
        sheet_name='Monthly Trend',
        index=False
    )

    city_failure.to_excel(
        writer,
        sheet_name='City Analysis',
        index=False
    )

    top_failed_merchants.to_excel(
        writer,
        sheet_name='Merchant Analysis',
        index=False
    )

    device_failure.to_excel(
        writer,
        sheet_name='Device Analysis',
        index=False
    )

    card_failure.to_excel(
        writer,
        sheet_name='Card Analysis',
        index=False
    )

    hourly_failure.to_excel(
        writer,
        sheet_name='Hourly Analysis',
        index=False
    )

In [ ]:
from google.colab import files

files.download(
    'FinTech_Payment_Analytics.xlsx'
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>